# 11 · RawKernel & 커널 개념 적용 기술

> **CuPy 2일 집중 코스 — Day 2 / 단원 8 (커널 심화)**

`RawKernel`은 **CUDA C 소스 전체**를 직접 작성하고 grid/block을 직접 지정하는 가장 낮은 수준의 방법입니다.
여기서 07의 커널 개념(인덱싱·coalescing·공유메모리·occupancy)을 **CUDA C로 직접 구현**하고,
커널을 빠르게 만드는 **개념 적용(최적화) 기술**을 정리합니다.

## 이 노트북에서 구현하는 개념 (07 참조)
- **2 인덱싱**(직접) · **5 coalescing** · **4 공유메모리(타일링/리덕션)** · **7 occupancy**(블록 튜닝)

## 학습 목표
- `RawKernel`로 1D·2D 커널을 작성하고 grid/block을 직접 지정한다.
- 공유메모리·`__syncthreads`로 **블록 리덕션**을 구현한다.
- 커널 최적화 기술(coalescing·타일링·분기 최소화·튜닝)을 적용한다.

> 본 과정 차별성('CUDA C 없이 Python만')상 RawKernel은 **심화/참고**입니다 — 같은 개념을 08~09는 Numba(Python)로 구현했습니다.

## 목차
1. [RawKernel 기초 (saxpy)](#1)
2. [2D 스텐실](#2)
3. [블록 크기 튜닝 (occupancy)](#3)
4. [커널 개념 적용(최적화) 기술](#4)
5. [공유메모리 블록 리덕션](#5)
6. [연습](#6)
7. [체크포인트](#7)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, allclose
print_env()

<a id="1"></a>
## 1. RawKernel 기초 (saxpy)

CUDA C로 `y = a*x + b` 를 작성합니다. 전역 인덱스(07의 2절)를 손으로 계산하고 경계를 검사합니다.
런치: `kernel((blocks,), (threads,), (args...))`.

In [ ]:
saxpy_src = r'''
extern "C" __global__
void saxpy(const float* x, float* y, float a, float b, int n){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = a * x[i] + b;
}'''
saxpy = cp.RawKernel(saxpy_src, 'saxpy')
n = 1 << 20
x = cp.random.rand(n, dtype=cp.float32); y = cp.empty_like(x)
threads = 256; blocks = (n + threads - 1) // threads
saxpy((blocks,), (threads,), (x, y, cp.float32(2.0), cp.float32(1.0), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(2.0*cp.asnumpy(x)+1.0, y, rtol=1e-5, atol=1e-5, name='saxpy')

<a id="2"></a>
## 2. 2D 스텐실

2D 인덱싱(`blockIdx/threadIdx`의 x·y)과 경계 처리를 직접 다룹니다. grid/block을 2D 튜플로 지정.

In [ ]:
stencil_src = r'''
extern "C" __global__
void stencil5(const float* x, float* y, int nx, int ny){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i <= 0 || j <= 0 || i >= nx-1 || j >= ny-1) return;
    int idx = j * nx + i;
    y[idx] = 0.25f*(x[idx-nx] + x[idx+nx] + x[idx-1] + x[idx+1]) - x[idx];
}'''
stencil5 = cp.RawKernel(stencil_src, 'stencil5')
nx, ny = 2048, 2048
x_np = np.random.rand(ny, nx).astype(np.float32)
def stencil_np(x):
    y = np.zeros_like(x)
    y[1:-1,1:-1] = 0.25*(x[:-2,1:-1]+x[2:,1:-1]+x[1:-1,:-2]+x[1:-1,2:]) - x[1:-1,1:-1]
    return y
ref = stencil_np(x_np)
xg = cp.asarray(x_np).ravel(); yg = cp.zeros_like(xg)
block = (16, 16); grid = (math.ceil(nx/16), math.ceil(ny/16))
stencil5(grid, block, (xg, yg, np.int32(nx), np.int32(ny)))
cp.cuda.Device().synchronize()
allclose(ref, yg.reshape(ny, nx), rtol=1e-5, atol=1e-5, name='stencil5')

<a id="3"></a>
## 3. 블록 크기 튜닝 (occupancy)

블록 모양에 따라 occupancy·메모리 효율이 달라져 성능이 변합니다(07의 7절). 직접 스윕해 최적점을 찾습니다.

In [ ]:
for block in [(8,8),(16,16),(32,8),(32,16)]:
    grid = (math.ceil(nx/block[0]), math.ceil(ny/block[1]))
    def run(g=grid, b=block): stencil5(g, b, (xg, yg, np.int32(nx), np.int32(ny)))
    print('block', block, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

## 3.1 전체 격자를 타일(Tile)로 쪼개는 개념
GPU는 2048 x 2048 크기의 거대한 격자 데이터를 블록(Block)이라는 작은 타일 형태로 나눕니다.
 ``` 
[ 전체 2048 x 2048 이미지 / 행렬 ]
+------------------------------------+
| Block(0,0) | Block(1,0) | ...      |
+------------+------------+          |
| Block(0,1) | Block(1,1) |          |
|    ...     |    ...     |          |
+------------------------------------+
  <--- Grid (전체 타일들의 집합) --->
```
Grid는 전체 작업 영역이고, Block은 GPU 코어 그룹(SM)에 배정되는 독립된 타일 단위입니다. 동일한 2048 x 2048 영역이라도 타일을 어떤 모양으로 자르느냐에 따라 GPU 내부의 연산 효율이 크게 달라집니다.

- 메모리 접근 방식의 차이 (핵심 요인: Memory Coalescing)
  * 성능 차이의 가장 큰 원인은 X축 크기(block.x)가 32냐 아니냐입니다. C/CUDA 메모리는 행(Row) 방향으로 연속 저장됩니다.
  * GPU의 최소 실행 단위인 Warp(32개 스레드)가 메모리를 읽을 때의 모습을 비교해서 그려주면 가장 직관적입니다.

```
block.x = 8 인 경우 (8x8)=
Warp (32개 스레드)가 8개씩 4줄로 쪼개짐:
[Row 0: 스레드 0~7  ] -> 연속된 8개 데이터 읽음 (32B)  ───┐
[Row 1: 스레드 8~15 ] -> 떨어진 위치 8개 읽음 (32B)    ├── 4번 나누어 메모리 요쳥!
[Row 2: 스레드 16~23] -> 떨어진 위치 8개 읽음 (32B)    │  (비효율적인 띄엄띄엄 접근)
[Row 3: 스레드 24~31] -> 떨어진 위치 8개 읽음 (32B)    ───┘
```
block.x = 32 인 경우 (32x8, 32x16)
Warp (32개 스레드)가 X축으로 1줄로 길게 정렬됨:
[Row 0: 스레드 0 ~ 31 ] -> 연속된 32개 데이터(128B)를 한 번에 통째로 로드!

- 블록당 총 스레드 수와 점유율 (Occupancy)
 * 블록 하나의 총 스레드 수(block.x * block.y)가 너무 적으면 GPU 연산 장치가 노는 시간(지연 시간)을 은닉하지 못합니다.

```
[ 블록 크기별 스레드 밀도 비교 ]

 (8, 8) = 64 Threads   │ (16, 16) = 256 Threads │ (32, 16) = 512 Threads
 +-------------------+ │ +--------------------+ │ +--------------------+
 | 2 Warps           | │ | 8 Warps            | │ | 16 Warps           |
 | (일감이 부족함)   | │ | (적절함)           | │ | (GPU 자원 꽉 채움) |
 +-------------------+ │ +--------------------+ │ +--------------------+
      3.34 ms                1.90 ms                 1.42 ms (최적!)
```

- 메모리 병합 접근(Coalescing)을 위해 X축 크기를 32의 배수(block.x = 32)로 지정해야 합니다.
- 지연 시간 은닉(Latency Hiding)을 위해 블록당 총 스레드 수(block.x * block.y)를 256 ~ 512개 수준으로 넉넉히 맞춰야 합니다.

<a id="4"></a>
## 4. 커널 개념 적용(최적화) 기술

07의 개념을 실제 커널에 적용하는 대표 기술입니다.

| 기술 | 적용 개념 | 효과 |
|------|-----------|------|
| **연속 접근**(grid-stride) | 5 coalescing | 메모리 트랜잭션↓·대역폭↑ |
| **공유메모리 타일링/리덕션** | 4 메모리계층 | 전역 접근↓·재사용↑ |
| **분기 최소화** | 3 SIMT | warp divergence↓ |
| **`const`/`__restrict__`** | — | 별칭 없음 가정 → 컴파일러 최적화 |
| **블록/스레드 튜닝** | 7 occupancy | 지연 숨김 |
| **연산 융합** | — | 중간배열·커널 런치↓ |

예: grid-stride 루프로 **연속 접근 + 임의 크기 처리**를 동시에 얻습니다.

In [ ]:
# grid-stride saxpy: warp 인접 스레드가 인접 주소 접근(coalesced), 큰 n도 처리
saxpy_gs_src = r'''
extern "C" __global__
void saxpy_gs(const float* __restrict__ x, float* __restrict__ y, float a, int n){
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;
    for (; i < n; i += stride) y[i] = a*x[i];
}'''
saxpy_gs = cp.RawKernel(saxpy_gs_src, 'saxpy_gs')
n = 1 << 24; x = cp.random.rand(n, dtype=cp.float32); y = cp.empty_like(x)
saxpy_gs((1024,), (256,), (x, y, cp.float32(3.0), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(3.0*cp.asnumpy(x), y, rtol=1e-5, atol=1e-5, name='saxpy_gs')

<a id="5"></a>
## 5. 공유메모리 블록 리덕션 (기술 종합)

합(reduction)을 **공유메모리 + `__syncthreads` + 트리 리덕션**으로 구현합니다 — 07의 개념 4·6을 RawKernel로 직접 적용.
각 블록이 부분합을 만들고, 블록 부분합을 마지막에 합칩니다. 공유메모리 크기는 런치 시 `shared_mem`으로 지정.

In [ ]:
reduce_src = r'''
extern "C" __global__
void block_sum(const float* __restrict__ x, float* partial, int n){
    extern __shared__ float sdata[];
    int tid = threadIdx.x;
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    sdata[tid] = (i < n) ? x[i] : 0.0f;   // coalesced 로드
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1){   // 트리 리덕션
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = sdata[0];
}'''
block_sum = cp.RawKernel(reduce_src, 'block_sum')
n = 1 << 22; x = cp.random.rand(n, dtype=cp.float32)
threads = 256; blocks = (n + threads - 1)//threads
partial = cp.empty(blocks, dtype=cp.float32)
block_sum((blocks,), (threads,), (x, partial, np.int32(n)), shared_mem=threads*4)
total = float(partial.sum())   # 블록 부분합 최종 합산
allclose(float(x.sum()), total, rtol=1e-3, atol=1e-1, name='block_sum')

<a id="6"></a>
## 6. 연습 — clamp RawKernel

`y = min(max(x, lo), hi)` 를 RawKernel(CUDA C)로 작성하세요(1D, 경계 검사·grid-stride 권장).

In [ ]:
import numpy as np
import cupy as cp
import math

# (이전 실습에서 사용한 allclose 함수가 있다고 가정합니다)
def allclose(a, b, rtol=1e-5, atol=1e-5, name=''):
    np.testing.assert_allclose(a, b, rtol=rtol, atol=atol)
    print(f"[allclose OK] {name}")

# 1. CUDA C RawKernel 작성 (Grid-Stride Loop 적용)
clamp_src = r'''
extern "C" __global__
void clamp(const float* __restrict__ x, float* __restrict__ y, float lo, float hi, int n){
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int stride = gridDim.x*blockDim.x;
    for (; i < n; i += stride){ float v = x[i]; v = v<lo?lo:(v>hi?hi:v); y[i] = v; }
}'''
# RawKernel 객체 생성
clamp = cp.RawKernel(clamp_src, 'clamp')

# 2. 테스트 데이터 생성
N = 1_000_000
x_np = np.random.randn(N).astype(np.float32)
ref = np.clip(x_np, -1.0, 1.0)

# GPU 메모리 할당
d_x = cp.asarray(x_np)
d_y = cp.empty_like(d_x)

# 3. Grid 및 Block 설정
block_dim = 256
#grid_dim = math.ceil(N / block_dim)
grid_dim = 1024

# 4. 커널 런치 (스칼라 값은 np.float32, np.int32로 타입 명시 필수!)
clamp(
    (grid_dim,), 
    (block_dim,), 
    (d_x, d_y, np.float32(-1.0), np.float32(1.0), np.int32(N))
)

# GPU 작업 완료 대기
cp.cuda.Device().synchronize()

# 5. 검증 (NumPy np.clip 결과와 비교)
allclose(ref, d_y.get(), rtol=1e-5, atol=1e-5, name='clamp')

#### grid_dim 숫자:
 * 데이터가 매우 커질수록(수천만~수억 개) Grid-Stride Loop와 '고정된 Grid 크기(예: 1024)'를 조합하는 것이 성능과 안정성 면에서 유리(NVIDIA 공식 권장 패턴)합니다.

#### 데이터 크기에 맞추는 방식 (ceil(N/256))
 * 데이터가 1,000만 개라면 약 39,000개의 블록이 생성됩니다. GPU는 이 블록들을 SM(GPU 코어 그룹)에 올렸다가, 계산이 끝나면 내리고, 다음 대기 중인 블록을 다시 올리는 과정을 수없이 반복해야 합니다. 
 * 마치 "택배 1개 배달할 때마다 단기 알바생을 새로 채용하고 해고하는 것"과 같아서 관리 비용(Overhead)이 발생합니다.

#### 고정된 Grid + Stride Loop (1024 등)
 * 블록을 딱 1024개만 만듭니다. 이 블록들이 GPU 하드웨어에 한 번 쫙 깔린 뒤, 커널 내부의 for 문을 돌면서 배열의 끝까지 데이터를 쭉쭉 빨아들이며 처리합니다. 즉, "정직원들을 고용해서 할당된 구역의 
 * 택배가 끝날 때까지 계속 일하게 하는 것"입니다. 스레드를 껐다 켜는 비용이 사라집니다.

#### 하드웨어 자원의 한계 (SM 포화 상태)
 * GPU의 실제 물리적 코어(SM) 개수는 한정되어 있습니다 (예: RTX 3090은 82개, 4090은 128개).
 * 어차피 GPU가 한 번에 동시에 돌릴 수 있는 블록의 수는 하드웨어 스펙에 의해 정해져 있습니다. 
 * 한 번에 기껏해야 몇백~천여 개의 블록만 동시에 돌아가기 때문에, 굳이 Grid를 수만~수십만 개로 쪼개서 GPU 스케줄러(작업 분배기)를 피곤하게 만들 필요가 없습니다. 
 * 하드웨어를 꽉 채울 정도(예: 1024개)만 던져주고, 나머지는 C++의 고속 for 루프에 맡기는 것이 스케줄러의 부하를 줄이는 비결입니다.

<a id="tiled"></a>
## (심화) 공유메모리 타일 transpose

전치(transpose)는 **쓰기가 비연속**이라 느립니다(08 naive). **공유메모리 타일**로 읽기·쓰기를 모두 연속(coalesced)으로 만듭니다:
타일을 공유메모리에 연속으로 읽어 들이고, `__syncthreads` 후 전치해서 연속으로 씁니다. (coalescing + 공유메모리 종합)
뱅크 충돌 회피를 위해 타일 폭을 `TILE+1`로 패딩합니다.

In [ ]:
transpose_src = r'''
#define TILE 32
extern "C" __global__
void transpose_tiled(const float* a, float* out, int n){
    __shared__ float tile[TILE][TILE+1];   // +1: 뱅크 충돌 회피
    int x = blockIdx.x*TILE + threadIdx.x;
    int y = blockIdx.y*TILE + threadIdx.y;
    if (x < n && y < n) tile[threadIdx.y][threadIdx.x] = a[y*n + x];  // 연속 읽기
    __syncthreads();
    int tx = blockIdx.y*TILE + threadIdx.x;
    int ty = blockIdx.x*TILE + threadIdx.y;
    if (tx < n && ty < n) out[ty*n + tx] = tile[threadIdx.x][threadIdx.y];  // 연속 쓰기
}'''
transpose_tiled = cp.RawKernel(transpose_src, 'transpose_tiled')
n = 2048; A = cp.random.rand(n, n, dtype=cp.float32); T = cp.empty_like(A)
block = (32, 32); grid = (n//32, n//32)
transpose_tiled(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(cp.asnumpy(A).T, T.reshape(n,n).get(), rtol=1e-5, atol=1e-5, name='tiled transpose')

**연습 — naive와 비교**: 08의 naive transpose(있으면)나 단순 버전과 타일 버전의 속도를 비교하세요(coalescing 효과).

In [ ]:
# 단순(비연속 쓰기) 비교용 RawKernel
naive_src = r'''
extern "C" __global__
void transpose_naive(const float* a, float* out, int n){
    int x = blockIdx.x*blockDim.x + threadIdx.x;
    int y = blockIdx.y*blockDim.y + threadIdx.y;
    if (x<n && y<n) out[x*n + y] = a[y*n + x];   // 쓰기 비연속
}'''
transpose_naive = cp.RawKernel(naive_src, 'transpose_naive')
def run_naive(): transpose_naive(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
def run_tiled(): transpose_tiled(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
print('naive', round(gpu_ms(bench(run_naive)),4), 'ms | tiled', round(gpu_ms(bench(run_tiled)),4), 'ms')

#### 그리드(Grid) -> GPU 전체 (모든 SM에 걸침)
  * 그리드는 우리가 지시한 '전체 작업(Workload)'입니다. 
  * GPU 스케줄러는 이 그리드 안에 있는 수많은 블록들을 GPU 기판에 있는 여러 SM(Streaming Multiprocessor)들에게 쫙 뿌려줍니다.
  * 즉, 그리드는 여러 SM을 가로지르며(across) 존재합니다.


#### 블록(Block) -> 단일 SM (절대 쪼개지지 않음)
  * 가장 중요한 규칙입니다. 하나의 블록은 반드시 단 하나의 SM에만 통째로 배정됩니다.
  * 예를 들어 스레드 1024개짜리 블록을 반으로 쪼개서 절반은 1번 SM에, 절반은 2번 SM에 넣는 일은 절대 불가능합니다.
  * 그 이유는 공유 메모리(Shared Memory) 때문입니다. 
  * 같은 블록에 속한 팀원들은 이 메모리를 같이 써야 하는데, 공유 메모리는 각 SM 내부에 물리적으로 고립된 하드웨어 칩이기 때문에 다른 SM에 있는 스레드와는 메모리를 공유할 수 없습니다.


#### SM 내부 -> 여러 개의 블록 동시 수용
  * 하나의 SM은 생각보다 덩치가 큽니다. 그래서 SM 내부에 여유 공간(공유 메모리 용량, 레지스터 등)이 허락하는 한, 여러 개의 블록을 동시에 SM 내부로 받아들여서 실행할 수 있습니다.


#### 하드웨어 매핑
  * Grid = 건물 전체 (GPU)에 배달할 택배 물량 전체
  * SM (Streaming Multiprocessor) = 각 층을 담당하는 작업 부서 (물리적 하드웨어)
  * Block = 한 부서(SM)에 배정되는 1개 조 (소프트웨어적 묶음)
  * Warp = 그 부서 안에서 실제로 32명씩 줄 맞춰서 일하는 행동 대원들 (실제 실행 단위)


#### 위의 예제에서 block = (32, 32); block = (32, 8); 로 했을때의 성능 비교
- block=(32, 32). 32 x 32 = 1024. 이는 GPU가 허용하는 최대 스레드 수입니다. 
- 32 x 32 크기의 데이터 타일 위에 1024명의 스레드를 1:1로 올려두고, 각자 자기 자리의 데이터만 딱 하나씩 처리하게 만듭니다.
- 혹은 block=(32, 8). 혹은 쓰레드 1024명을 관리하는 것도 오버헤드다. 똘똘한 256명만 뽑아서 4배로 일하게 하자!"

<a id="7"></a>
## 7. 체크포인트

- [ ] RawKernel로 1D(saxpy)·2D(스텐실) 커널을 작성·런치했다
- [ ] 블록 크기 튜닝으로 occupancy 영향을 봤다
- [ ] 최적화 기술(coalescing·타일링·분기·튜닝)을 안다
- [ ] 공유메모리+`__syncthreads`로 블록 리덕션을 구현했다

다음: **`12_interop_frameworks`** — DLPack으로 PyTorch 등과 무복사 연동합니다.